In [0]:
customers = spark.read.csv('/Volumes/workspace/default/olist_data/olist_customers_dataset.csv', header=True, inferSchema=True)

orders = spark.read.csv('/Volumes/workspace/default/olist_data/olist_orders_dataset.csv', header=True, inferSchema=True)

order_items = spark.read.csv('/Volumes/workspace/default/olist_data/olist_order_items_dataset.csv', header=True, inferSchema=True)

products = spark.read.csv('/Volumes/workspace/default/olist_data/olist_products_dataset.csv', header=True, inferSchema=True)

customers.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import sum, col, rank
from pyspark.sql.window import Window

# Step 1: Join tables
df = orders.join(customers, "customer_id") \
           .join(order_items, "order_id")


In [0]:
customer_spend = df.groupBy("customer_id", "customer_city") \
    .agg(sum("price").alias("total_spend"))

customer_spend.show(5)

+--------------------+--------------------+-----------+
|         customer_id|       customer_city|total_spend|
+--------------------+--------------------+-----------+
|9ef432eb625129730...|           sao paulo|      29.99|
|cf8ffeddf027932e5...|santa barbara d'o...|      179.0|
|e8a332c3433fbd379...|        porto alegre|      29.99|
|0dd5a932b1e29daa1...|             socorro|       28.9|
|69d8d9775287ba592...|         sao goncalo|     178.99|
+--------------------+--------------------+-----------+
only showing top 5 rows


In [0]:
window = Window.partitionBy("customer_city").orderBy(col("total_spend").desc())

ranked_df = customer_spend.withColumn("rank", rank().over(window))

ranked_df.show(5)

+--------------------+-------------------+-----------+----+
|         customer_id|      customer_city|total_spend|rank|
+--------------------+-------------------+-----------+----+
|9e01f714a2b3b8962...|abadia dos dourados|      199.0|   1|
|a23e3f9a2b656b23b...|abadia dos dourados|      120.0|   2|
|f11eb8f0b8b87510a...|abadia dos dourados|       39.9|   3|
|576d71ddb21b21763...|          abadiania|     949.99|   1|
|d47c8bb51df6f716e...|             abaete|      449.0|   1|
+--------------------+-------------------+-----------+----+
only showing top 5 rows


In [0]:
top3 = ranked_df.filter(col("rank") <= 3)

top3.show()

+--------------------+-------------------+-----------+----+
|         customer_id|      customer_city|total_spend|rank|
+--------------------+-------------------+-----------+----+
|9e01f714a2b3b8962...|abadia dos dourados|      199.0|   1|
|a23e3f9a2b656b23b...|abadia dos dourados|      120.0|   2|
|f11eb8f0b8b87510a...|abadia dos dourados|       39.9|   3|
|576d71ddb21b21763...|          abadiania|     949.99|   1|
|d47c8bb51df6f716e...|             abaete|      449.0|   1|
|ff0d62f8be4c098e6...|             abaete|      225.9|   2|
|5371894984937a27c...|             abaete|      208.9|   3|
|c7eb06383ae604616...|         abaetetuba|     1500.0|   1|
|367fd22f1e994de87...|         abaetetuba|      797.6|   2|
|7727e2cc9ec428ad3...|         abaetetuba|     580.27|   3|
|f8508e9ec506a046a...|            abaiara|      169.0|   1|
|9723d86b12bdec025...|            abaiara|       93.9|   2|
|11888d3334232a1eb...|             abaira|      145.0|   1|
|43b0018187e283119...|             abair

In [0]:
from pyspark.sql.functions import to_date

orders = orders.withColumn(
    "order_date",
    to_date("order_purchase_timestamp")
)

In [0]:
df_sales = orders.join(order_items, "order_id")

In [0]:
from pyspark.sql.functions import sum

daily_sales = df_sales.groupBy("order_date") \
    .agg(sum("price").alias("daily_sales")) \
    .orderBy("order_date")

daily_sales.show()

+----------+------------------+
|order_date|       daily_sales|
+----------+------------------+
|2016-09-04|             72.89|
|2016-09-05|              59.5|
|2016-09-15|            134.97|
|2016-10-02|             100.0|
|2016-10-03|            463.48|
|2016-10-04| 9940.960000000003|
|2016-10-05|           8343.25|
|2016-10-06| 7960.510000000001|
|2016-10-07|           7228.05|
|2016-10-08|           8441.85|
|2016-10-09|3336.9899999999993|
|2016-10-10|           3692.57|
|2016-12-23|              10.9|
|2017-01-05|             396.9|
|2017-01-06|            916.38|
|2017-01-07|            1351.9|
|2017-01-08|            709.58|
|2017-01-09|            673.79|
|2017-01-10|           1434.87|
|2017-01-11|2776.1600000000003|
+----------+------------------+
only showing top 20 rows


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col

window = Window.orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, 0)

running_total_df = daily_sales.withColumn(
    "running_total",
    sum("daily_sales").over(window)
)

running_total_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+------------------+------------------+
|order_date|       daily_sales|     running_total|
+----------+------------------+------------------+
|2016-09-04|             72.89|             72.89|
|2016-09-05|              59.5|            132.39|
|2016-09-15|            134.97|            267.36|
|2016-10-02|             100.0|            367.36|
|2016-10-03|            463.48|            830.84|
|2016-10-04| 9940.960000000003|10771.800000000003|
|2016-10-05|           8343.25|19115.050000000003|
|2016-10-06| 7960.510000000001|27075.560000000005|
|2016-10-07|           7228.05| 34303.61000000001|
|2016-10-08|           8441.85| 42745.46000000001|
|2016-10-09|3336.9899999999993|46082.450000000004|
|2016-10-10|           3692.57|49775.020000000004|
|2016-12-23|              10.9|49785.920000000006|
|2017-01-05|             396.9| 50182.82000000001|
|2017-01-06|            916.38|51099.200000000004|
|2017-01-07|            1351.9|52451.100000000006|
|2017-01-08|            709.58|

In [0]:
from pyspark.sql.functions import sum

product_sales = order_items.groupBy("product_id") \
    .agg(sum("price").alias("total_sales"))

product_sales.show(5)

+--------------------+-----------------+
|          product_id|      total_sales|
+--------------------+-----------------+
|4244733e06e7ecb49...|            533.1|
|89321f94e35fc6d79...|3375.709999999999|
|e19ddcc85537b41f2...|629.7900000000001|
|2d9ff06c8870a518f...|            156.0|
|54e5063e43f27f747...|            152.0|
+--------------------+-----------------+
only showing top 5 rows


In [0]:
product_with_category = product_sales.join(
    products, "product_id"
)

product_with_category.select(
    "product_id", "product_category_name", "total_sales"
).show(5)

+--------------------+---------------------+-----------------+
|          product_id|product_category_name|      total_sales|
+--------------------+---------------------+-----------------+
|4244733e06e7ecb49...|           cool_stuff|            533.1|
|89321f94e35fc6d79...|            alimentos|3375.709999999999|
|e19ddcc85537b41f2...|     moveis_decoracao|629.7900000000001|
|2d9ff06c8870a518f...| utilidades_domest...|            156.0|
|54e5063e43f27f747...| industria_comerci...|            152.0|
+--------------------+---------------------+-----------------+
only showing top 5 rows


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

window = Window.partitionBy("product_category_name") \
    .orderBy(col("total_sales").desc())

ranked_products = product_with_category.withColumn(
    "rank",
    dense_rank().over(window)
)

ranked_products.select(
    "product_category_name", "product_id", "total_sales", "rank"
).show()

+---------------------+--------------------+------------------+----+
|product_category_name|          product_id|       total_sales|rank|
+---------------------+--------------------+------------------+----+
|                 NULL|5a848e4ab52fd5445...|24229.029999999984|   1|
|                 NULL|eed5cbd74fac3bd79...|            9945.0|   2|
|                 NULL|b1d207586fca400a2...|            7152.0|   3|
|                 NULL|76d1a1a9d21ab677a...|           5712.64|   4|
|                 NULL|ad88641611c35ebd5...| 4515.330000000001|   5|
|                 NULL|3b60d513e90300a4e...|           4393.33|   6|
|                 NULL|4c50dcc50f1512f46...|            3980.0|   7|
|                 NULL|17823ffd2de8234f0...|           2969.89|   8|
|                 NULL|0e030462875259ec0...|            2740.0|   9|
|                 NULL|b36f3c918c91478c4...|            2550.0|  10|
|                 NULL|f58e45b16a42a325c...|           2359.91|  11|
|                 NULL|a14aea6067b

In [0]:
df_clv = orders.join(order_items, "order_id")

In [0]:
from pyspark.sql.functions import sum

clv = df_clv.groupBy("customer_id") \
    .agg(sum("price").alias("total_spend"))

clv.show()

+--------------------+-----------+
|         customer_id|total_spend|
+--------------------+-----------+
|cf8ffeddf027932e5...|      179.0|
|241e78de29b3090cf...|      113.0|
|e1862648f338ecaf4...|      37.99|
|3c628393675b42c6b...|      359.8|
|328d7a69cb9cbaf08...|       90.0|
|aa601b3c45980c091...|      49.99|
|d4c5a2a1316f738e7...|       96.0|
|ae8db0691449a4435...|      109.9|
|4d225460f83ea2b1a...|      89.99|
|af5b08db5f1a31a3b...|      39.99|
|1ae196062dab95e43...|       39.0|
|fbf8e675c5adcf8c4...|      109.9|
|5613454932b2c4229...|       40.0|
|94e56a6d6f8b3e26a...|      119.0|
|f31eb05ebdef1ede7...|      170.0|
|b7919647bde69acc9...|      168.0|
|638c6674418fc5828...|      225.0|
|cf3c35f463f6a234d...|      109.8|
|661652a4ba9255d6a...|       24.9|
|5ed8d20445c356efa...|      32.64|
+--------------------+-----------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import when, col

segmented_df = clv.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
)

segmented_df.show()

+--------------------+-----------+-------+
|         customer_id|total_spend|segment|
+--------------------+-----------+-------+
|cf8ffeddf027932e5...|      179.0| Bronze|
|241e78de29b3090cf...|      113.0| Bronze|
|e1862648f338ecaf4...|      37.99| Bronze|
|3c628393675b42c6b...|      359.8| Bronze|
|328d7a69cb9cbaf08...|       90.0| Bronze|
|aa601b3c45980c091...|      49.99| Bronze|
|d4c5a2a1316f738e7...|       96.0| Bronze|
|ae8db0691449a4435...|      109.9| Bronze|
|4d225460f83ea2b1a...|      89.99| Bronze|
|af5b08db5f1a31a3b...|      39.99| Bronze|
|1ae196062dab95e43...|       39.0| Bronze|
|fbf8e675c5adcf8c4...|      109.9| Bronze|
|5613454932b2c4229...|       40.0| Bronze|
|94e56a6d6f8b3e26a...|      119.0| Bronze|
|f31eb05ebdef1ede7...|      170.0| Bronze|
|b7919647bde69acc9...|      168.0| Bronze|
|638c6674418fc5828...|      225.0| Bronze|
|cf3c35f463f6a234d...|      109.8| Bronze|
|661652a4ba9255d6a...|       24.9| Bronze|
|5ed8d20445c356efa...|      32.64| Bronze|
+----------

In [0]:
segment_count = segmented_df.groupBy("segment").count()

segment_count.show()

+-------+-----+
|segment|count|
+-------+-----+
| Bronze|98660|
| Silver|    5|
|   Gold|    1|
+-------+-----+



In [0]:
from pyspark.sql.functions import count

order_count = orders.groupBy("customer_id") \
    .agg(count("*").alias("total_orders"))

order_count.show()

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|9ef432eb625129730...|           1|
|456dc10730fbdba34...|           1|
|52142aa69d8d0e124...|           1|
|2a1dfb647f32f4390...|           1|
|4f28355e5c17a4a42...|           1|
|148348ff65384b424...|           1|
|8628fac2267e8c880...|           1|
|1833a0540067becaf...|           1|
|df5c9e02596851fcb...|           1|
|62ffae18a7ca4b2e6...|           1|
|6feea03756fd7ef4f...|           1|
|2d8748fb35d51c2fd...|           1|
|3a017955f82b5bf61...|           1|
|f7398fc942c8fa80e...|           1|
|a51b373c427132a77...|           1|
|9e5ce657315b2bdb9...|           1|
|b7ee9b824b4a83b38...|           1|
|eb1c0e1ca3e17ecca...|           1|
|b3519c0352b07bc30...|           1|
|3f7d26944f7f68bd2...|           1|
+--------------------+------------+
only showing top 20 rows


In [0]:
customer_info = customers.select("customer_id", "customer_city")

final_df = clv.join(customer_info, "customer_id") \
              .join(order_count, "customer_id")

In [0]:
from pyspark.sql.functions import when, col

final_df = final_df.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
)

In [0]:
final_df = final_df.select(
    "customer_id",
    "customer_city",
    "total_spend",
    "segment",
    "total_orders"
)

final_df.show()

+--------------------+--------------------+-----------+-------+------------+
|         customer_id|       customer_city|total_spend|segment|total_orders|
+--------------------+--------------------+-----------+-------+------------+
|816f8653d5361cbf9...|           sao paulo|       49.0| Bronze|           1|
|a9d37ddc8ba4d9f6d...|           joinville|       27.9| Bronze|           1|
|bb2f5e670f7155dc6...|           sao paulo|       50.0| Bronze|           1|
|388025bec8128ff20...|            cascavel|      190.0| Bronze|           1|
|818596f5b68adfe2c...|      rio de janeiro|       63.6| Bronze|           1|
|4d225460f83ea2b1a...|           cariacica|      89.99| Bronze|           1|
|0fa5382628eea6bbb...|      alfredo chaves|      22.32| Bronze|           1|
|d1c6e90ab127d42b7...|         avanhandava|     179.99| Bronze|           1|
|d29935fdaf4a76f34...|            confresa|      189.9| Bronze|           1|
|2fdffca8dcdf01547...|             itatiba|      109.0| Bronze|           1|